In [4]:
import numpy as np
import time

## Week 2 code


In [ ]:
# week 1 code refined


def build_element_connectivity_matrix(n_elements):
    connectivity_array = []
    print(
        "\nEnter the end nodes for each element separated by space (for e.g.: '1 2')."
    )
    for i in range(n_elements):
        ends = input(f"Element {i + 1}: ")
        end_nodes_array = [int(end) for end in ends.split(" ")]
        end_nodes_array.sort()
        connectivity_array.append(end_nodes_array)
    return connectivity_array


def build_stiffness_vector(n_elements):
    stiffness_vector = []
    print("\nEnter the stiffness coefficient for each element.")
    for i in range(n_elements):
        stiffness = float(input(f"Element {i + 1}: "))
        stiffness_vector.append(stiffness)
    return stiffness_vector


def build_local_stiffness_matrices_spring(stiffness_vector):
    n_elements = len(stiffness_vector)
    local_stiffness_matrices = []
    for i in range(n_elements):
        local_stiffness_matrices.append(
            np.array(
                [
                    [stiffness_vector[i], -stiffness_vector[i]],
                    [-stiffness_vector[i], stiffness_vector[i]],
                ]
            )
        )
    return np.array(local_stiffness_matrices)


def build_global_stiffness_matrix_from_local_stiffness_matrices(
    n_nodes, local_stiffness_matrices, connectivity_array
):
    n_elements = local_stiffness_matrices.shape[0]
    global_stiffness_matrix = np.zeros((n_nodes, n_nodes))
    for i in range(n_elements):
        nodes = connectivity_array[i]
        for j in range(len(nodes)):
            for k in range(len(nodes)):
                global_stiffness_matrix[nodes[j] - 1, nodes[k] - 1] += (
                    local_stiffness_matrices[i, j, k]
                )
    return global_stiffness_matrix


def build_force_vector(n_nodes):
    force_vector = []
    print(
        "\nEnter the force on each node. Simply press enter (without content) if unknown."
    )
    for i in range(n_nodes):
        force = input(f"Node {i + 1}: ")
        force_vector.append(float(force)) if force != "" else force_vector.append(None)
    return force_vector


def build_displacement_vector(n_nodes):
    displacement_vector = []
    print(
        "\nEnter the displacement on each node. Simply press enter (without content) if unknown."
    )
    for i in range(n_nodes):
        displacement = input(f"Node {i + 1}: ")
        displacement_vector.append(
            float(displacement)
        ) if displacement != "" else displacement_vector.append(None)
    return displacement_vector


# Identify unknowns
def identify_unknowns(force_vector, displacement_vector):
    unknown_list = []
    for i in range(len(force_vector)):
        if force_vector[i] == None and displacement_vector[i] == None:
            raise Exception(f"Node {i + 1} is an unknown.")
        if force_vector[i] == None:
            unknown_list.append(f"F_{i + 1}")
        elif displacement_vector[i] == None:
            unknown_list.append(f"u_{i + 1}")
    return unknown_list


def build_new_system_equations(
    force_vector, global_stiffness_matrix, displacement_vector, unknown_list
):
    # y = A x
    new_y = np.zeros(len(force_vector))
    new_A = np.zeros((len(force_vector), len(force_vector)))

    for i in range(len(force_vector)):
        if force_vector[i] != None:
            new_y[i] = force_vector[i]
        else:
            new_A[i, i] = 1
        for j in range(len(displacement_vector)):
            if displacement_vector[j] != None:
                new_y[i] -= global_stiffness_matrix[i, j] * displacement_vector[j]
            else:
                new_A[i, j] = global_stiffness_matrix[i, j]
    return new_y, new_A


def refined_FEM_direct_stiffness_method():
    n_elements = int(input("Enter the number of elements:"))
    n_nodes = int(input("Enter the number of nodes:"))
    connectivity_array = build_element_connectivity_matrix(n_elements)
    stiffness_vector = build_stiffness_vector(n_elements)
    local_stiffness_matrices = build_local_stiffness_matrices_spring(stiffness_vector)
    global_stiffness_matrix = (
        build_global_stiffness_matrix_from_local_stiffness_matrices(
            n_nodes, local_stiffness_matrices, connectivity_array
        )
    )
    force_vector = build_force_vector(n_nodes)
    displacement_vector = build_displacement_vector(n_nodes)

    print(f"\n======== SYSTEM DETAILS ========\n")
    print(f"Number of elements: {n_elements}")
    print(f"Number of nodes: {n_nodes}")
    print(
        f"\nConnectivity array: {[f'Element{i + 1}: {connectivity_array[i]}' for i in range(len(connectivity_array))]}"
    )
    print(
        f"\nLocal stiffness matrices: {[f'Element{i + 1}:\n{local_stiffness_matrices[i]}' for i in range(len(local_stiffness_matrices))]}"
    )
    print(f"\nGlobal stiffness matrix:\n{global_stiffness_matrix}")

    print(f"\n======== SOLVING THE SYSTEM ========\n")
    print(f"(00.00 ms) Compiling unknown list and rearranging equations...")
    start_time = time.time()
    unknown_list = identify_unknowns(force_vector, displacement_vector)
    new_y, new_A = build_new_system_equations(
        force_vector, global_stiffness_matrix, displacement_vector, unknown_list
    )
    print(f"({(time.time() - start_time) * 1000:.2f} ms) Solving system... ")
    solved_unknown = np.linalg.solve(new_A, new_y)
    print(f"({(time.time() - start_time) * 1000:.2f} ms) Printing results...")
    for i in range(len(unknown_list)):
        print(f"{unknown_list[i]} = {solved_unknown[i]}")

## Generalized code

In [ ]:
def build_local_stiffness_matrices_spring(stiffness_vector):
    n_elements = len(stiffness_vector)
    local_stiffness_matrices = []
    for i in range(n_elements):
        local_stiffness_matrices.append(
            np.array(
                [
                    [stiffness_vector[i], -stiffness_vector[i]],
                    [-stiffness_vector[i], stiffness_vector[i]],
                ]
            )
        )
    return np.array(local_stiffness_matrices)


def build_local_stiffness_matrices_bar(A_vector, E_vector, L_vector):
    n_elements = len(A_vector)
    coeff_vector = [A_vector[i] * E_vector[i] / L_vector[i] for i in range(n_elements)]
    local_stiffness_matrices = []
    for i in range(n_elements):
        local_stiffness_matrices.append(
            np.array(
                [
                    [coeff_vector[i], -coeff_vector[i]],
                    [-coeff_vector[i], coeff_vector[i]],
                ]
            )
        )
    return np.array(local_stiffness_matrices)


def build_local_stiffness_matrices_truss(
    A_vector, E_vector, L_vector, theta_deg_vector
):
    n_elements = len(A_vector)
    coeff_vector = [A_vector[i] * E_vector[i] / L_vector[i] for i in range(n_elements)]
    cos_theta = np.cos(np.deg2rad(theta_deg_vector))
    sin_theta = np.sin(np.deg2rad(theta_deg_vector))
    local_stiffness_matrices = []
    for i in range(n_elements):
        local_stiffness_matrices.append(
            np.array(
                [
                    [
                        cos_theta[i] ** 2 * coeff_vector[i],
                        cos_theta[i] * sin_theta[i] * coeff_vector[i],
                        -(cos_theta[i] ** 2) * coeff_vector[i],
                        -cos_theta[i] * sin_theta[i] * coeff_vector[i],
                    ],
                    [
                        cos_theta[i] * sin_theta[i] * coeff_vector[i],
                        sin_theta[i] ** 2 * coeff_vector[i],
                        -cos_theta[i] * sin_theta[i] * coeff_vector[i],
                        -(sin_theta[i] ** 2) * coeff_vector[i],
                    ],
                    [
                        -(cos_theta[i] ** 2) * coeff_vector[i],
                        -cos_theta[i] * sin_theta[i] * coeff_vector[i],
                        cos_theta[i] ** 2 * coeff_vector[i],
                        cos_theta[i] * sin_theta[i] * coeff_vector[i],
                    ],
                    [
                        -cos_theta[i] * sin_theta[i] * coeff_vector[i],
                        -(sin_theta[i] ** 2) * coeff_vector[i],
                        cos_theta[i] * sin_theta[i] * coeff_vector[i],
                        sin_theta[i] ** 2 * coeff_vector[i],
                    ],
                ]
            )
        )

    return np.array(local_stiffness_matrices)


def build_global_stiffness_matrix_from_local_stiffness_matrices(
    n_nodes, local_stiffness_matrices, connectivity_array
):
    n_elements = local_stiffness_matrices.shape[0]
    global_stiffness_matrix = np.zeros((n_nodes, n_nodes))
    for i in range(n_elements):
        nodes = connectivity_array[i]
        for j in range(len(nodes)):
            for k in range(len(nodes)):
                global_stiffness_matrix[nodes[j] - 1, nodes[k] - 1] += (
                    local_stiffness_matrices[i, j, k]
                )
    return global_stiffness_matrix


def FEM_direct_stiffness_method_general(system_txt_file=None, only_save=False):

    if system_txt_file is not None:
        # Read system from txt file
        try:
            with open(system_txt_file, "r") as f:
                lines = f.readlines()
        except FileNotFoundError:
            print(f"Searching systems folder for '{system_txt_file}'...")
            with open(f"systems/{system_txt_file}", "r") as f:
                lines = f.readlines()
        variables = {}
        for line in lines:
            if line.startswith("##"):
                var_name, var_type = line[2:].strip().split(" ")
                variables[var_name] = {"type": var_type, "value": []}
            elif not line.startswith("#"):
                if var_name in variables:
                    if variables[var_name]["type"] == "int":
                        variables[var_name]["value"] = int(line.strip())
                    elif variables[var_name]["type"] == "float":
                        variables[var_name]["value"] = float(line.strip())
                    elif variables[var_name]["type"] == "str":
                        variables[var_name]["value"] = line.strip()
                    elif variables[var_name]["type"].startswith("list"):
                        if line.strip() != "":
                            if variables[var_name]["type"] == "list_of_lists_int":
                                variables[var_name]["value"].append(
                                    [int(x) for x in line.strip()[1:-1].split(",")]
                                )
                            elif variables[var_name]["type"] == "list_float":
                                variables[var_name]["value"].append(float(line.strip()))
                            elif variables[var_name]["type"] == "list_float_or_None":
                                value = line.strip()
                                if value == "None":
                                    variables[var_name]["value"].append(None)
                                else:
                                    variables[var_name]["value"].append(float(value))
        n_elements = variables["n_elements"]["value"]
        n_nodes = variables["n_nodes"]["value"]
        connectivity_array = variables["connectivity_array"]["value"]
        element_type = variables["element_type"]["value"]
        if element_type == "spring":
            stiffness_vector = variables["stiffness_vector"]["value"]
            local_stiffness_matrices = build_local_stiffness_matrices_spring(
                stiffness_vector
            )
        elif element_type == "bar":
            A_vector = variables["A_vector"]["value"]
            E_vector = variables["E_vector"]["value"]
            L_vector = variables["L_vector"]["value"]
            local_stiffness_matrices = build_local_stiffness_matrices_bar(
                A_vector, E_vector, L_vector
            )
        elif element_type == "truss":
            A_vector = variables["A_vector"]["value"]
            E_vector = variables["E_vector"]["value"]
            L_vector = variables["L_vector"]["value"]
            theta_deg_vector = variables["theta_deg_vector"]["value"]
            local_stiffness_matrices = build_local_stiffness_matrices_truss(
                A_vector, E_vector, L_vector, theta_deg_vector
            )
        global_stiffness_matrix = (
            build_global_stiffness_matrix_from_local_stiffness_matrices(
                n_nodes, local_stiffness_matrices, connectivity_array
            )
        )

        force_vector = variables["force_vector"]["value"]
        displacement_vector = variables["displacement_vector"]["value"]
    else:
        n_elements = int(input("Enter the number of elements:"))
        n_nodes = int(input("Enter the number of nodes:"))
        connectivity_array = build_element_connectivity_matrix(n_elements)

        # -- Enter element properties
        element_type = input("Enter the element type (spring, bar, truss):")
        if element_type == "spring":
            stiffness_vector = []
            print("\nEnter the stiffness coefficient for each element.")
            for i in range(n_elements):
                stiffness = float(input(f"Element {i + 1}: "))
                stiffness_vector.append(stiffness)
            local_stiffness_matrices = build_local_stiffness_matrices_spring(
                stiffness_vector
            )
        elif element_type == "bar":
            A_vector = []
            E_vector = []
            L_vector = []
            print(
                "\nEnter the cross-sectional area, Young's modulus and length of each element. (e.g.: '1 2 3')"
            )
            for i in range(n_elements):
                A, E, L = input(f"Element {i + 1}: ").split(" ")
                A_vector.append(float(A))
                E_vector.append(float(E))
                L_vector.append(float(L))
            local_stiffness_matrices = build_local_stiffness_matrices_bar(
                A_vector, E_vector, L_vector
            )
        elif element_type == "truss":
            A_vector = []
            E_vector = []
            L_vector = []
            theta_deg_vector = []
            print(
                "\nEnter the cross-sectional area, Young's modulus, length and angle of each element. (e.g.: '1 2 3 4')"
            )
            for i in range(n_elements):
                A, E, L, theta = input(f"Element {i + 1}: ").split(" ")
                A_vector.append(float(A))
                E_vector.append(float(E))
                L_vector.append(float(L))
                theta_deg_vector.append(float(theta))
            local_stiffness_matrices = build_local_stiffness_matrices_truss(
                A_vector, E_vector, L_vector, theta_deg_vector
            )

        # -- Build global stiffness matrix

        global_stiffness_matrix = (
            build_global_stiffness_matrix_from_local_stiffness_matrices(
                n_nodes, local_stiffness_matrices, connectivity_array
            )
        )

        # -- Enter Forces
        force_vector = []
        print(
            "\nEnter the force on each node. Simply press enter (without content) if unknown."
        )
        for i in range(n_nodes):
            force = input(f"Node {i + 1}: ")
            force_vector.append(float(force)) if force != "" else force_vector.append(
                None
            )

        # -- Enter displacements
        displacement_vector = []
        print(
            "\nEnter the displacement on each node. Simply press enter (without content) if unknown."
        )
        for i in range(n_nodes):
            displacement = input(f"Node {i + 1}: ")
            displacement_vector.append(
                float(displacement)
            ) if displacement != "" else displacement_vector.append(None)

    print(f"\n\n======== SYSTEM DETAILS ========\n")
    print(f"Number of elements: {n_elements}")
    print(f"Number of nodes: {n_nodes}")
    print(
        f"\nConnectivity array: {[f'Element{i + 1}: {connectivity_array[i]}' for i in range(len(connectivity_array))]}"
    )
    for i in range(len(local_stiffness_matrices)):
        print(f"\nElement {i + 1} local stiffness matrix:")
        print(local_stiffness_matrices[i])
    print(f"\nGlobal stiffness matrix:")
    print(global_stiffness_matrix)

    if system_txt_file is None:
        print(f"\n\n======== SAVING THE SYSTEM ========\n")
        trial_name = input("Enter a name for the trial (for saving): ")
        if trial_name == "":
            pass
        else:
            with open(f"systems/{trial_name}.txt", "w") as f:
                # header
                f.write(f"# =================== {trial_name} ===================\n")
                f.write(f"# ----------------------------------------------------\n")
                f.write(
                    f"# Lines starting with '#' are comments and will be ignored when reading the file.\n"
                )
                f.write(
                    f"# Lines starting with '##' are variable names and types, followed by their values which will be read when reading the file.\n"
                )
                f.write(f"# In case of values for multiple elements / nodes, enter them one line after another (for e.g.: A_vector, E_vector, L_vector, theta_deg_vector, force_vector, displacement_vector)\n")
                f.write(f"# In case of multiple values for a single element / node, enter them in a list format (for e.g.: connectivity_array)\n")
                f.write(f"# ----------------------------------------------------\n")
                f.write(f"## n_elements int\n")
                f.write(f"{n_elements}\n")
                f.write(f"## n_nodes int\n")
                f.write(f"{n_nodes}\n")
                f.write(f"## connectivity_array list_of_lists_int\n")
                for i in range(len(connectivity_array)):
                    f.write(f"{connectivity_array[i]}\n")
                f.write(f"## element_type str\n")
                f.write(f"{element_type}\n")
                if element_type == "spring":
                    f.write(f"## stiffness_vector list_float\n")
                    for i in range(len(stiffness_vector)):
                        f.write(f"{stiffness_vector[i]}\n")
                elif element_type == "bar":
                    f.write(f"## A_vector list_float\n")
                    for i in range(len(A_vector)):
                        f.write(f"{A_vector[i]}\n")
                    f.write(f"## E_vector list_float\n")
                    for i in range(len(E_vector)):
                        f.write(f"{E_vector[i]}\n")
                    f.write(f"## L_vector list_float\n")
                    for i in range(len(L_vector)):
                        f.write(f"{L_vector[i]}\n")
                elif element_type == "truss":
                    f.write(f"## A_vector list_float\n")
                    for i in range(len(A_vector)):
                        f.write(f"{A_vector[i]}\n")
                    f.write(f"## E_vector list_float\n")
                    for i in range(len(E_vector)):
                        f.write(f"{E_vector[i]}\n")
                    f.write(f"## L_vector list_float\n")
                    for i in range(len(L_vector)):
                        f.write(f"{L_vector[i]}\n")
                    f.write(f"## theta_deg_vector list_float\n")
                    for i in range(len(theta_deg_vector)):
                        f.write(f"{theta_deg_vector[i]}\n")
                f.write(f"## force_vector list_float_or_None\n")
                for i in range(len(force_vector)):
                    f.write(f"{force_vector[i]}\n")
                f.write(f"## displacement_vector list_float_or_None\n")
                for i in range(len(displacement_vector)):
                    f.write(f"{displacement_vector[i]}\n")
                # footer
                f.write(f"# ======================= END =======================\n")

    if only_save:
        print(f"\n\n======== SYSTEM SAVED ========\n")
        return
    print(f"\n\n======== SOLVING THE SYSTEM ========\n")
    print(f"(00.00 ms) Compiling unknown list and rearranging equations...")
    start_time = time.time()
    unknown_list = identify_unknowns(force_vector, displacement_vector)
    print(f"({(time.time() - start_time) * 1000:.2f} ms) Unknowns: {unknown_list}")
    new_y, new_A = build_new_system_equations(
        force_vector, global_stiffness_matrix, displacement_vector, unknown_list
    )
    print(f"({(time.time() - start_time) * 1000:.2f} ms) Solving system... ")
    solved_unknown = np.linalg.solve(new_A, new_y)
    print(f"({(time.time() - start_time) * 1000:.2f} ms) Printing results...")
    print(f"\n\n======== RESULTS ========\n")
    for i in range(len(unknown_list)):
        print(f"\t{unknown_list[i]} = {solved_unknown[i]}")


### test spring

In [ ]:
FEM_direct_stiffness_method_general()

### test truss

In [12]:
# Problem from class
# (one with two truss elements, one horizontal and one slanted, one of both fxed to wall and the other together)

# given values
A_1 = 1250  # mm2
A_2 = 1000  # mm2
L_2 = 750  # mm
dist_btw_ends_left = 500  # mm
theta_2_deg = 0  # deg
E_1 = 200 * 10 ^ 3  # MPa
E_2 = 200 * 10 ^ 3  # MPa
P = 1  # ---- VAR

# some calculations
L_1 = np.sqrt(L_2**2 + dist_btw_ends_left**2)
theta_1_deg = np.rad2deg(np.arctan(500 / 750))
print(f"L_1 = {L_1:.2f}")
print(f"theta_1_deg = {theta_1_deg:.2f}")

# -- other details
# 2 elements
# 6 d.o.f.
# 1,2 at node 1    |       3,4 at node 2    |    5,6 at node 3
# node 1 and 2 are fixed, three is at which elements meet
# P perpendicularly down at node 3, for F_5 = 0, F_6 = -P

L_1 = 901.39
theta_1_deg = 33.69


In [13]:
FEM_direct_stiffness_method_general(system_txt_file="truss_qn_class.txt")

Searching systems folder for 'truss_qn_class.txt'...


======== SYSTEM DETAILS ========

Number of elements: 2
Number of nodes: 6

Connectivity array: ['Element1: [1, 2, 5, 6]', 'Element2: [3, 4, 5, 6]']

Element 1 local stiffness matrix:
[[ 192011.44349285  128007.30212462 -192011.44349285 -128007.30212462]
 [ 128007.30212462   85337.98350323 -128007.30212462  -85337.98350323]
 [-192011.44349285 -128007.30212462  192011.44349285  128007.30212462]
 [-128007.30212462  -85337.98350323  128007.30212462   85337.98350323]]

Element 2 local stiffness matrix:
[[ 266666.66666667       0.         -266666.66666667      -0.        ]
 [      0.               0.              -0.              -0.        ]
 [-266666.66666667      -0.          266666.66666667       0.        ]
 [     -0.              -0.               0.               0.        ]]

Global stiffness matrix:
[[ 192011.44349285  128007.30212462       0.               0.
  -192011.44349285 -128007.30212462]
 [ 128007.30212462   85337.9835

In [ ]:
# given solutions

K_1 = (
    np.array([[9, 6, -9, -6], [6, 4, -6, -4], [-9, -6, 9, 6], [-6, -4, 6, 4]])
    * 10**6
    * (1 / (13 * np.sqrt(13)))
)
K_2 = (
    np.array([[1, 0, -1, 0], [0, 0, 0, 0], [-1, 0, 1, 0], [0, 0, 0, 0]])
    * 10**5
    * (8 / 3)
)

print(f"K_1 (Element 1)")
print(K_1)
print(f"\nK_2 (Element 2)")
print(K_2)